In [24]:
import os
import pandas as pd
import numpy as np

---
## Bagian G — Diterapkan pada proyek tim (dikerjakan di luar kelas)

Bagian ini adalah isi **M1**. Kerjakan bersama tim pada dataset proyek Anda sendiri.

### G1 — Pertanyaan analisis
Tulis **3–5 pertanyaan** yang lolos tiga uji. Untuk setiap pertanyaan, sebutkan kolom yang akan dipakai.

### Pertanyaan payung
Bagaimana pola penjualan buah dan sebaran pendapatan di Toko Buah Segar Utama selama periode Januari–Maret 2019?

#### Pertanyaan Turunan:

1. Kategori produk (*Product Line*) apa yang menghasilkan total keuntungan bersih (*Gross Income*) terbesar di Cabang A selama Q1 2019? Menggunakan kolom Product Line dan Gross Income
2. Bagaimana perbandingan total volume penjualan item (*Quantity*) berdasarkan tipe pelanggan (*Customer Type*) dan gender? Menggunakan kolom Quantity, Customer Type, dan Gender
3. Bagaimana tren total pendapatan harian (*Total*) di cabang tersebut sepanjang periode Januari–Maret 2019? Menggunakan kolom Date dan Total
4. Metode pembayaran (*Payment*) apa yang paling sering digunakan oleh pelanggan untuk bertransaksi? Menggunakan kolom Payment


### G2 — Provenans data
Catat lima butir berikut di README repositori tim:

| Butir | Isi |
|---|---|
| Nama sumber | Kaggle |
| Tautan atau nama instansi | https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales |
| Tanggal pengambilan | Kamis, 01 Oktober 2026 |
| Lisensi atau nomor surat izin | Apache 2.0 |
| Kalimat sitasi untuk laporan | Data sekunder mengenai transaksi penjualan ritel yang dianalisis dalam laporan ini bersumber dari dataset publik 'Supermarket Sales' yang diunggah oleh kontributor Fares Ashraf di platform Kaggle dan didistribusikan di bawah lisensi Apache 2.0.|

### G3 — Kamus data
Isi `TEMPLATE_Kamus_Data.csv` untuk **setiap** kolom dataset tim, termasuk kolom `kategori_data_pribadi` dan `tindakan_penanganan`. Simpan sebagai `data_dictionary.md` atau `.csv` di repositori.

In [25]:
# Import dataset
FILE = 'SuperMarket Analysis.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df_team = pd.read_csv(FILE)

print(f"Dimensi data SuperMarket Analysis: {df_team.shape[0]} baris dan {df_team.shape[1]} kolom")
print(df_team.info())
print(df_team.dtypes)

Dimensi data SuperMarket Analysis: 1000 baris dan 17 kolom
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Sales                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                

In [26]:
# Membuat kamus data
kamus_team = pd.DataFrame({
    'nama_kolom': df_team.columns,
    'tipe_data': [str(df_team[col].dtype) for col in df_team.columns],
    'nilai_unik': [df_team[col].nunique() for col in df_team.columns],
    'nilai_kosong': [df_team[col].isnull().sum() for col in df_team.columns],
    'contoh_nilai': [df_team[col].dropna().iloc[0] if not df_team[col].dropna().empty else np.nan for col in df_team.columns]
})

# Membuat object untuk pemetaan kategori data per kolom
penandaan_team = {
    'Invoice ID': ('pengenal langsung', 'hash'),

    'Branch': ('quasi-identifier', 'pertahankan'),
    'City': ('quasi-identifier', 'pertahankan'),
    'Customer type': ('quasi-identifier', 'pertahankan'),
    'Gender': ('quasi-identifier', 'pertahankan'),
    'Date': ('quasi-identifier', 'pertahankan'),
    'Time': ('quasi-identifier', 'pertahankan'),

    'Product line': ('bukan data pribadi', 'pertahankan'),
    'Unit price': ('bukan data pribadi', 'pertahankan'),
    'Quantity': ('bukan data pribadi', 'pertahankan'),
    'Tax 5%': ('bukan data pribadi', 'pertahankan'),
    'Sales': ('bukan data pribadi', 'pertahankan'),
    'Payment': ('bukan data pribadi', 'pertahankan'),
    'cogs': ('bukan data pribadi', 'pertahankan'),
    'gross margin percentage': ('bukan data pribadi', 'pertahankan'),
    'gross income': ('bukan data pribadi', 'pertahankan'),
    'Rating': ('bukan data pribadi', 'pertahankan'),
}

list_tanda = []
for kolom, (kategori, tindakan) in penandaan_team.items():
  list_tanda.append({
      'nama_kolom': kolom,
      'kategori': kategori,
      'tindakan': tindakan
  })

tanda_team = pd.DataFrame(list_tanda)

# Memetakan setiap kategori data pada DataFrame kamus data
kamus_team['tindakan_penanganan'] = kamus_team['nama_kolom'].map(
    tanda_team.set_index('nama_kolom')['tindakan']
).fillna('pertahankan')

In [27]:
# Fungsi hash
import hashlib

def samarkan(nilai, garam):
  if pd.isna(nilai):
    return np.nan

  teks_gabungan = str(garam) + str(nilai)
  teks_bytes = teks_gabungan.encode('utf-8')

  hasil_sha256 = hashlib.sha256(teks_bytes).hexdigest()
  return hasil_sha256[:12]

# Fungsi cek_k
def cek_k(data, kolom):
  ukuran_group = data.groupby(kolom, observed=True).size()

  k = ukuran_group.min()
  group_1_orang = (ukuran_group == 1).sum()
  total_group = len(ukuran_group)

  return pd.Series([k, group_1_orang, total_group], ['k', 'group_1_orang', 'total_group'])

In [28]:
# G4 — jalankan penanganan pada data tim dan periksa nilai k
# Buat DataFrame baru bernama 'aman_team' yang sudah membuang semua kolom bertindakan 'hapus'.
kolom_dihapus = kamus_team[kamus_team['tindakan_penanganan'] == 'hapus']['nama_kolom'].tolist()

aman_team = df_team.drop(columns=kolom_dihapus)

kolom_dihash = kamus_team[kamus_team['tindakan_penanganan'] == 'hash']['nama_kolom'].tolist()

# men-enkripsi kolom bertindakan 'hash'
from google.colab import userdata
GARAM = userdata.get('GARAM')

for col in kolom_dihash:
  if col in aman_team.columns:
    aman_team[col] = aman_team[col].apply(lambda x: samarkan(x, GARAM))
print(f"Contoh hasil hash kolom 'Invoice ID': {aman_team['Invoice ID'][0]}")

nilai_k = cek_k(aman_team, ['Branch', 'City', 'Customer type', 'Gender'])
print(nilai_k)
# Salin pola Bagian E dan F, ganti dataset dan nama kolomnya.
# Simpan hasilnya ke data/processed/ dan JANGAN unggah data/raw/
aman_team.to_csv("SuperMarket Analysis_cleaned.csv", index=False)
print("Status: Berkas 'SuperMarket Analysis_cleaned.csv' aman tersimpan.")



Contoh hasil hash kolom 'Invoice ID': 57c45d0ae03d
k                62
group_1_orang     0
total_group      12
dtype: int64
Status: Berkas 'SuperMarket Analysis_cleaned.csv' aman tersimpan.


### G5 — Daftar periksa sebelum `git push`

- [ ] `data/raw/` masuk `.gitignore`
- [ ] Tidak ada nama, NIK, nomor telepon, surel, atau alamat di berkas mana pun
- [ ] Keluaran sel yang sempat menampilkan data pribadi sudah dibersihkan
- [ ] Garam hash tidak tertulis di dalam notebook
- [ ] Kamus data memuat kategori dan tindakan untuk setiap kolom
- [ ] Nilai k pada data yang diunggah sudah dihitung dan dicatat
- [ ] Bila masih ragu: repositori dibuat **privat**

> Menghapus berkas lalu *commit* ulang **tidak** menghilangkan data dari riwayat Git. > Periksa **sebelum** *commit* pertama.